In [15]:
import pandas as pd
import numpy as np

In [16]:
df = pd.read_excel('dados/base_preliminar2.xlsx')


sinan_mg = pd.read_csv('dados/sinan_mg2.csv', delimiter=';')

/tmp/ipykernel_11451/1808962648.py:4: DtypeWarning: Columns (0: COM_RENAL, 1: COM_EDEMA, 2: COM_SEPTIC, 3: COM_CHOQUE, 4: DT_OBITO, 5: DT_DIGITA) have mixed types. Specify dtype option on import or set low_memory=False.
  sinan_mg = pd.read_csv('dados/sinan_mg2.csv', delimiter=';')


## Variáveis relacionadas à população

In [17]:
# Criar categoria de tempo
df['CAT_TEMPO'] = pd.cut(x=df['TEMPO'],
       bins=[0, 30, 60, float('inf')],
       labels=['< 30', '30 a 59', '60+'],
       include_lowest=True)

# Criar proporção de população 0 a 10 em relação à população geral
df['PROP_POP_0A10'] = round(df['POP10'] / df['POP_GERAL'],2)

# Criar proporção de população 60+ em relação a população geral
df['PROP_POP_60MAIS'] = round(df['POP60'] / df['POP_GERAL'],2)

# Log populacional
df['LOG_POP'] = np.log(df['POP_GERAL'])

## Variáveis relacionadas à incidência

In [18]:
# Criar proporção de MG em relação ao Total de casos
df['PROP_MG'] = round(df['MG'] / df['TOTAL_CASOS'],2)

# Incidência média geral por 100 mil
df['INCID_MEDIA_GERAL'] = round(((df['TOTAL_CASOS']/5)/df['POP_GERAL'])*100000,2)

# Incidência média anual por 100 mil
df["INC_MEDIA_ANUAL"] = (df["TOTAL_CASOS"] / (df["POP_GERAL"] * 6)) * 100000

# Incidência média 0a10 anos por 100 mil
df['INCID_0A10'] = round(((df['TOTAL_10A']/5)/df['POP10'])*100000,2)

# Incidência média 60+ anos por 100 mil
df['INCID_60MAIS'] = round(((df['TOTAL_60']/5)/df['POP60'])*100000,2)

# Proporção de casos 0a10 em relação ao total de casos
df['PROP_CASOS_0A10'] = round(df['TOTAL_10A']/df['TOTAL_CASOS'],4)

# Proporção de casos 60+ em relação ao total de casos
df['PROP_CASOS_60MAIS'] = round(df['TOTAL_60']/df['TOTAL_CASOS'],4)

# Taxa MG por 100 mil
df['TAXA_MG_100MIL'] = round(((df['MG']/5)/df['POP_GERAL'])*100000,2)


In [19]:
## Ignorados

# Classificação Ignorados
class_ig = sinan_mg[sinan_mg['TRA_CLASSI'].isna()]
class_ig = class_ig['ID_MN_RESI'].value_counts().reset_index(name='CLASS_IG')

# Soro Ignorados
soro_ig = sinan_mg[sinan_mg['CON_SOROTE'].isna()]
soro_ig = soro_ig['ID_MN_RESI'].value_counts().reset_index(name='SORO_IG')

# Evolução Ignorados
evol_ig = sinan_mg[sinan_mg['EVOLUCAO'].isna()]
evol_ig =  evol_ig['ID_MN_RESI'].value_counts().reset_index(name = 'EVOL_IG')


ignorados = [class_ig,soro_ig,evol_ig]



## Merge dos ignorados
for i in ignorados:
    
    df = df.merge(
        how='left',
        right=i,
        left_on='IBGE',
        right_on='ID_MN_RESI'
    )

df = df.drop(columns=['ID_MN_RESI_x', 'ID_MN_RESI_y', 'ID_MN_RESI'])

In [20]:
## Proporções dos ignorados


# Definir zeros
cols = [
    'CLASS_IG',
    'SORO_IG',
    'EVOL_IG'
]

df[cols] = df[cols].fillna(0)




# Proporção de Classificação Ignorada em relação ao total de casos
df['PROP_CLASS_IGN'] = round(df['CLASS_IG']/df['TOTAL_CASOS'],4)

# Proporção de Soroterapia Ignorada em relação ao total de casos
df['PROP_SORO_IGN'] = round(df['SORO_IG']/df['TOTAL_CASOS'],4)

# Proporção de Evolução Ignorada em relação ao total de casos
df['PROP_EVOL_IGN'] = round(df['EVOL_IG']/df['TOTAL_CASOS'],4)


## Perfil de distância

In [21]:
import numpy as np

df["CAT_TEMPO"] = np.select(
    [
        df["ACESSO_LOCAL"] == 1,

        (df["ACESSO_LOCAL"] == 0) &
        (df["TEMPO"] <= 30),

        (df["ACESSO_LOCAL"] == 0) &
        (df["TEMPO"] > 30) &
        (df["TEMPO"] <= 60),

        (df["ACESSO_LOCAL"] == 0) &
        (df["TEMPO"] > 60)
    ],
    [
        "Local",
        "≤30 min",
        "31–60 min",
        ">60 min"
    ],
    default=None
)



ordem_tempo = [
    "Local",
    "≤30 min",
    "31–60 min",
    ">60 min"
]

df["CAT_TEMPO"] = pd.Categorical(
    df["CAT_TEMPO"],
    categories=ordem_tempo,
    ordered=True
)

In [22]:
# Variavel de estabilidade (0 ou 1)
df['BAIXO_N'] = (df['TOTAL_CASOS'] < 10).astype(int)

In [23]:
# Renomeia colunas e exclui uma que nao interessa
df = df.drop(columns='Unnamed: 0')
df.rename(columns={'MUNI_REFERENCIADO': 'MUNI'}, inplace=True)

In [24]:
# Variavel de Categorias de tempo
df["TEMPO_50"] = pd.cut(
    df["TEMPO"],
    bins=[-np.inf, 50, np.inf],
    labels=["≤50 min", ">50 min"]
)

df.TEMPO_50.value_counts()

TEMPO_50
≤50 min    404
>50 min     21
Name: count, dtype: int64

In [25]:
# Carga epidemilógica MG
df["CARGA_MG"] = pd.cut(
    df["MG"],
    bins=[-np.inf, 2, 6, np.inf],
    labels=["Baixa", "Intermediária", "Alta"]
)
df['CARGA_MG'].value_counts()

CARGA_MG
Alta             284
Baixa            202
Intermediária    159
Name: count, dtype: int64

In [26]:
# Checagem
pd.crosstab(df['TEMPO_50'], df['CARGA_MG'])

CARGA_MG,Baixa,Intermediária,Alta
TEMPO_50,,,
≤50 min,142,128,134
>50 min,13,3,5


In [27]:
df.to_excel('dados/base_analise.xlsx')